# **TensorRT**

---

## What TensorRT Is

**TensorRT** is NVIDIA’s inference SDK that compiles a trained network into a highly optimized **engine** specialized for a *specific* GPU. It is an ahead-of-time optimizer + runtime, not a training framework. Its main wins:

* **Layer & tensor fusion** — e.g. fuse conv + bias + ReLU into one kernel, cutting memory traffic and launch overhead.
* **Reduced precision** — run in **FP16** or quantized **INT8**, often 2–4× faster on tensor-core GPUs.
* **Kernel auto-tuning** — benchmarks multiple kernel implementations for *your* GPU and picks the fastest.
* **Static memory planning** — reuses buffers to shrink the runtime footprint.

The trade-off: the resulting engine is tied to the GPU architecture, TensorRT version, and (often) a fixed input shape range it was built for.

## Precision Modes

| Mode | Speed | Accuracy cost | Notes |
|------|-------|---------------|-------|
| **FP32** | baseline | none | numerically identical to training |
| **FP16** | ~2× | usually negligible | enable with `enabled_precisions={torch.half}`; needs tensor-core GPU |
| **INT8** | ~2–4× | small–moderate | requires **calibration** on representative data to compute per-tensor scales |

INT8 needs a calibration dataset because it maps float ranges to 8-bit integers; pick calibration data that matches production inputs or accuracy will drop.

## Two Paths into TensorRT

**Path A — Torch-TensorRT** (compile directly from PyTorch, stay in Python):

```python
import torch, torch_tensorrt

model = MyModel().eval().cuda()
example = torch.randn(1, 3, 224, 224).cuda()

trt_model = torch_tensorrt.compile(
    model,
    inputs=[torch_tensorrt.Input((1, 3, 224, 224), dtype=torch.float32)],
    enabled_precisions={torch.float16},   # allow FP16 kernels
)
out = trt_model(example)                  # runs the optimized engine
torch_tensorrt.save(trt_model, "model_trt.ep", inputs=[example])
```

**Path B — ONNX → TensorRT engine** (export once, build with `trtexec` or the Python API):

```bash
# 1) export from PyTorch to ONNX (see the ONNX/TorchScript notebook)
# 2) build an FP16 engine from the ONNX file:
trtexec --onnx=model.onnx --fp16 --saveEngine=model.engine \
        --minShapes=input:1x3x224x224 \
        --optShapes=input:8x3x224x224 \
        --maxShapes=input:16x3x224x224
```

Use the explicit `min/opt/maxShapes` when you need **dynamic batch size**; otherwise the engine is locked to one shape.

## When to Use TensorRT (and When Not)

**Use it when:**

* you deploy on NVIDIA GPUs and latency/throughput is the bottleneck,
* the model is mostly standard CNN/Transformer ops TensorRT supports,
* the GPU/driver/TensorRT version in production is fixed (engines are not portable across them).

**Skip / reconsider when:**

* you serve on CPU or non-NVIDIA hardware (use ONNX Runtime / OpenVINO instead),
* the model has many custom or unsupported ops (will fall back and lose the gains),
* build time and engine-version brittleness outweigh the speedup for low-traffic services.

Always re-run **parity tests** after conversion: FP16/INT8 change numerics, so compare engine outputs against the original within a tolerance before shipping.

## References

* Torch-TensorRT: https://pytorch.org/TensorRT/
* TensorRT developer guide: https://docs.nvidia.com/deeplearning/tensorrt/latest/index.html
* `trtexec`: https://docs.nvidia.com/deeplearning/tensorrt/latest/inference-library/command-line-programs.html
* INT8 calibration: https://docs.nvidia.com/deeplearning/tensorrt/latest/inference-library/work-quantized-types.html